**MLP USING NUMPY**

In [ ]:
# Libraries
import numpy as np
import random
import matplotlib.pyplot as plt

Normalizing the pixel values so that all features are scaled equally. 

In [ ]:
def z_normalize_images(images):
    mean = images.mean()
    std  = images.std()
    eps  = 1e-8
    return (images - mean) / (std + eps)

The model we are building is a multi-layer perceptron. The networks architecture is defined in __init__. The input dimension is 2304 (42x42 pixels flattened), stemming from the input images. 5 layers feed the input forward, meaning that 5 weight matrices are used to convert the inputs. Every layer has a dimension of 64, meaning 64 neurons are built into each layer. Every output neuron equates one class that an input can be interpreted as. Here, we deal with 7 classes, one for each facial expression. 

ReLu is used as an activation function. This allows for non-linearity in the output.

Softmax normalizes logits to turn them into probabilities.

The last function defines the forward process. All layers except for the last one apply ReLU, while the last one returns logits. During the training, softmax will be applied on these logits. This allows to make precise predictions that are easy to understand even for a human observer. Poor performance can easily be detected. 
When the model is actually predicted, we don´t really care about the probabilities. All the model is supposed to deliver is one class. Skipping softmax will save computational power, as in order to find the predicted class, nothing but finding the largest logit is needed. 

Finally, the model backpropagates the losses by calculating all partial derivatives of each weight matrix and applying the chain rule to propagate the updates backwards through the layers. 

In [ ]:
class MLP ():
    #input layer is not counted in n_layers
    #output layer is
    def __init__(self, input_dim=2304, n_layers=5, hidden_dim=64, n_classes=7):
        rng = np.random.default_rng(seed=42) 
        
        self.n_layers = n_layers
        self.hidden_dim = hidden_dim
        self.layers = []
        for i in range (n_layers):
            in_dim = hidden_dim
            out_dim = hidden_dim
            if i == 0:
                in_dim = input_dim
            elif (i+1) == n_layers:
                out_dim = n_classes
            std = 0.2
            if in_dim == out_dim:
                std = np.sqrt(2/in_dim)
            current_layer = rng.normal(
                #mean
                loc=0.0,      
                #standard deviation
                scale=std,        
                size=(in_dim, out_dim)
            ).astype(np.float32)
            self.layers.append (current_layer)

    def ReLU (self, x):
        x = np.asarray(x)
        return np.maximum(0, x)

    def _get_normalized_logits_with_softmax_denom(self, logits):
        logits = logits - np.max(logits)
        exp_logits = np.exp(logits)
        softmax_denominator = np.sum(exp_logits)
        return exp_logits / softmax_denominator, softmax_denominator

    """
    def _get_normalized_logits_with_softmax_denom(self, logits):
        softmax_denominator = np.sum(np.exp(logits))
        return np.exp(logits) / softmax_denominator, softmax_denominator
    """

    def forward(self, inputs, target_value=None, requires_grad=False):
        hidden_layer_activations = []
        #layer_activations.append(inputs.copy())
     
        
        for i in range(self.n_layers):
            #ReLU activations in all the layers but the last
            if i == 0:
                hidden_layer_activations.append(self.ReLU(inputs @ self.layers[i]))
            elif (i+1) < self.n_layers:
                hidden_layer_activations.append(self.ReLU(hidden_layer_activations[i-1] @ self.layers[i]))
            #no activation function applied so far
            #in the last layer
            #softmax will be applied to the output 
            #of the last layer later if necessary
            else:
                logits = hidden_layer_activations[i-1] @ self.layers[i]

        #if no target value is passed to the model.forward
        #then the model is in the inference mode
        #no logit/output normalization/softmax is necessary
        #in the inference mode
        #since one can base model prediction on the highest
        #unnormalized/unsoftmaxed logit
        if target_value is None:
            return logits
        else:
            #this is softmax
            #softmax denominator is returned by softmax 
            #on top of normalized logits to be
            #reused in computing CEL
            normalized_logits, softmax_denom = self._get_normalized_logits_with_softmax_denom(logits)

            
            #CEL is equal to -ln(exp(logits[target_value])/softmax_denom))
            #the formula below is algebraically equivalent to the one above
            #CEL_value = -logits[target_value] + np.log(softmax_denom)
            # normalized logits for stable computation
            CEL_value = -np.log(normalized_logits[target_value])
            
        #if no gradient is required,
        #then just return CEL
        if not requires_grad:
            return CEL_value
        

        #if a target value is passed to model.forward
        #and CEL is required
        #then the model is in the training mode
        #one needs to do softmax on the inputs
        #to pass softmaxed logits into 
        #CEL/cross-entropy loss
        else:

            #this one is the gradient of CEL
            #d_softmax = normalized_logits.copy()
            d_softmax = normalized_logits
            d_softmax[target_value] -= 1

            #initialize an list with as many empty
            #elements as there are hidden layers
            #i.e. param matrices
            layer_gradients = [None] * self.n_layers

            #this computes gradients of layer params
            for i in range(self.n_layers-1, -1, -1):
                #output softmax layer
                if i == (self.n_layers-1):
                    #this is the part of the gradient
                    #which is reused across layers
                    dynamic_gradient = d_softmax

                    #this one contrains gradient of i-th layer
                    layer_gradients[i] = np.outer(hidden_layer_activations[i-1], dynamic_gradient)
                else:
                    #this one multiplies dynamic gradient by the gradient of pre-activations
                    #gradient of pre-activations is equal to the corresponding weight matrix
                    #which us stored in self.layers[i+1]
                    dynamic_gradient = self.layers[i+1] @ dynamic_gradient
                    #gradient of hidden ReLU activation is equal to 1
                    #if that activation is equal to 1 and 0 otherwise 
                    relu_grad = (hidden_layer_activations[i]>0).astype(float)
                    #mulitply dynamic grad by activation gradient
                    dynamic_gradient *= relu_grad
                    #input layer
                    if i == 0:
                        layer_gradients[i] = np.outer(inputs, dynamic_gradient)
                    #neither output nor input layer
                    else:
                        layer_gradients[i] = np.outer(hidden_layer_activations[i-1], dynamic_gradient)
            return CEL_value, layer_gradients

The next function runs teh model on a dataset and computes the overall loss: 

In [ ]:
def evaluate_model_on(model, dataset):
    total_loss = 0
    for x, y in dataset:
        total_loss += model.forward(x, y, requires_grad=False)
    return total_loss/len(dataset)

Next up, the model parameters must be determined. We are doing this using gradient descent, looping over each sample individually.

In [ ]:
def train_model_with_SGD (model, 
                         training_set,
                         validation_set,
                         lr: float, 
                         n_epochs: int, 
                         sgd_lr_multiplier: float = 0.95
                        ):
    

    print ("_" * 50)
    print (f"Initial LR = {lr}")
    print (f"LR multipliter per epoch = {sgd_lr_multiplier:.5f}")
    print (f"Number of layers = {model.n_layers}")
    print (f"Dimensionality of hidden layers = {model.hidden_dim}")
    print ("_" * 50)
    train_loss_history = []
    val_loss_history = []

    for epoch_index in range(1, n_epochs + 1):

        print (f"Epoch {epoch_index}/{n_epochs}")
        print (f"current SGD learning rate = {lr}")
        total_train_loss = 0

        #shuffle training set in a reproducible manner
        random.seed(42 + epoch_index)
        random.shuffle(training_set) 

        for x,y in training_set:

            #get the CEL gradient from the forward pass directly
            loss, layer_grads = model.forward(x, y, requires_grad=True)
         
            #do SGD step
            for i in range(model.n_layers):
                #grad_norm = np.linalg.norm(layer_grads[i], ord=2)
                #if grad_norm > 1:
                #    layer_grads[i] /= grad_norm
                model.layers[i] -= lr*layer_grads[i]

            total_train_loss += loss
          
        #decrease lr each epoch
        lr *= sgd_lr_multiplier
        
        #compute, print and save avg loss per epoch
        avg_train_loss = total_train_loss / len(training_set)
        print (f"average train loss = {avg_train_loss:.5f}")
        train_loss_history.append(avg_train_loss)
 
        avg_val_loss = evaluate_model_on (model, validation_set)
        print (f"average val loss = {avg_val_loss:.5f}")
        val_loss_history.append(avg_val_loss)
        print ("_" * 50)

    return model, train_loss_history, val_loss_history

In [ ]:
def plot_loss(train_loss_history, val_loss_history):
    plt.figure(figsize=(8, 5))
    plt.plot(train_loss_history, label="Training Loss")
    plt.plot(val_loss_history, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Average Loss")
    plt.title("Training vs Validation Loss")
    plt.legend()
    plt.grid(True)
    plt.show()


Grid search tries out several combinations of hyperparameters in order to achieve the lowest validation loss. 

In [ ]:
def grid_search(hyperparameters: dict,
                train_set: list,
                validation_set: list,
                n_epochs: int):
    best_loss = float("inf")
    best_params = {}
    best_model = None

    # extract hyperparameter combinations
    learning_rate = hyperparameters.get('lr', [])
    lr_multipliers = hyperparameters.get('lr_multiplier', [])
    hidden_dims = hyperparameters.get('hidden_dim', [])
    n_layers_list = hyperparameters.get('n_layers', [])

    # iterate over all combinations
    for lr in learning_rate:
        for lr_multiplier in lr_multipliers:
            for hidden_dim in hidden_dims:
                for n_layers in n_layers_list:
                    print(f"Testing: lr={lr}, lr_multiplier={lr_multiplier}, "
                          f"hidden_dim={hidden_dim}, n_layers={n_layers}")

                    # create a new model for each combination
                    model = MLP(n_layers=n_layers, hidden_dim=hidden_dim)

                    # train the model and track validation loss history
                    _, _, val_loss_history = train_model_with_SGD(
                        model,
                        train_set,
                        validation_set,
                        lr=lr,
                        n_epochs=n_epochs,
                        sgd_lr_multiplier=lr_multiplier
                    )

                    # find the best validation loss
                    min_val_loss = min(val_loss_history)
                    print (f"min_val_loss = {min_val_loss:.5f}")

                    # check if new combination is best
                    if min_val_loss < best_loss:
                        best_loss = min_val_loss
                        best_params = {
                            'lr': lr,
                            'lr_multiplier': lr_multiplier,
                            'hidden_dim': hidden_dim,
                            'n_layers': n_layers
                        }
                        best_model = model

    return best_model, best_params, best_loss

Splitting datasets into validation and training data, input and target values. The data is shuffled with a fixed seed in order to take out any unwanted data structure that may arise from grouped or sorted data. INputs are normalized and decimal notation is specified.

In [ ]:
train_x = np.load('train_mlp_x.npy') 
train_y = np.load('train_mlp_y.npy')
val_and_test_x  = np.load('test_mlp_x.npy')   
val_and_test_y  = np.load('test_mlp_y.npy')

N = len(val_and_test_x)
perm = np.random.RandomState(seed=42).permutation(N)
split_at = N // 2

val_idx = perm[:split_at]
test_idx = perm[split_at:]

val_x  = val_and_test_x[val_idx]
val_y  = val_and_test_y[val_idx]
test_x = val_and_test_x[test_idx]
test_y = val_and_test_y[test_idx]

# z_normalization of inputs
train_x  = z_normalize_images(train_x)  
val_x  = z_normalize_images(val_x)  
test_x  = z_normalize_images(test_x)  

#create training, val and test set by zipping 
#corresponding inputs and targets
training_set = list(zip(train_x, train_y))
val_set = list(zip(val_x, val_y))
test_set = list(zip(test_x, test_y))

#numpy printing instruction for decimal notation
np.set_printoptions(
    precision   = 5,       
    floatmode   = 'fixed',  
    suppress    = True     
)


Hyperparameters to be tested are defined - 27 combinations of hyperparameters are possible, meaning 27 MLPs ared evaluated using grid search. The best results are printed.

In [ ]:
#mlp = MLP()

#SGD_LEARNING_RATE = 2e-3
#LEARNING_RATE_MULTIPLIER_PER_EPOCH = 0.95
# Define the hyperparameter grid
hyperparameters_to_tune = {
    'lr': [0.001, 0.0005, 0.0001],
    'lr_multiplier': [0.99],
    'hidden_dim': [32, 64, 128],
    'n_layers': [4, 7, 10]
    }
N_EPOCHS = 10

best_model, best_params, best_loss = grid_search(
        hyperparameters_to_tune,
        training_set,
        val_set,
        n_epochs=N_EPOCHS
    )

print("Best hyperparameters found:")
print(f"Learning rate: {best_params['lr']}")
print(f"LR multiplier: {best_params['lr_multiplier']}")
print(f"Hidden dim: {best_params['hidden_dim']}")
print(f"Number of layers: {best_params['n_layers']}")
print(f"Best validation loss: {best_loss:.5f}")

Let´s plot this: 

In [ ]:
model, train_loss_history, val_loss_history = train_model_with_SGD(
    model, training_set, validation_set, lr=0.01, n_epochs=20
)

plot_loss(train_loss_history, val_loss_history)

The final evaluation is performed, calculating the overall cross entropy loss. 

In [ ]:
# Evaluate the best model on test set
avg_test_loss = evaluate_model_on(best_model, test_set)
print(f"TEST LOSS = {avg_test_loss:.5f}")
print("_" * 50)

"""
mlp, train_loss_history_SGD, val_loss_history_SGD = train_model_with_SGD (mlp,
                                            list(training_set),
                                            list(val_set),
                                            SGD_LEARNING_RATE,
                                            N_EPOCHS,
                                            LEARNING_RATE_MULTIPLIER_PER_EPOCH
                                            )
"""